In [56]:
#Importing necessary libraries
import cv2
import numpy as np
from matplotlib import pyplot as plt
import pickle
import gzip
import bitarray


#Node Structure/class

class Node:
   def __init__(self, key=None,freq=None):
      self.freq = freq
      self.key = key
      self.left = None
      self.right = None

#1.Read it to memory from the file





# img_box_filtered = np.zeros((rows,cols,dims),np.uint8)


#2.Making dictionary for the pixel-values and frequency of that pixel-value
def pixelDict(img):
    pixels_dict=dict()
    rows,cols,dims = img.shape
    for i in range(rows):
        for j in range(cols):
            for k in range(dims):
                if img[i,j,k] in pixels_dict:
                    pixels_dict[img[i,j,k]]+=1
                else:
                    pixels_dict[img[i,j,k]]=1
    return pixels_dict

#3.Add all values into min-heap
def make_minHeap(pixels_dict):
  min_heap=[0]
  for key in pixels_dict.keys():
      newNode = Node(key,pixels_dict[key])
      min_heap.append(newNode)
  return min_heap

#min-heap

def PARENT(i):
 return int(i/2)

def LEFT(i):
 return 2*i

def RIGHT(i):
 return 2*i + 1

#inser to min-heap
def insertToMinHeap(A,x):
  A.append(x)
  i = len(A)-1
  while(i>1 and A[PARENT(i)].freq > A[i].freq):
    temp = A[PARENT(i)]
    A[PARENT(i)]= A[i]
    A[i]=temp
    i=PARENT(i)


#min-heapify
def minHeapify(A,i):
    L=LEFT(i)
    r=RIGHT(i)

    if L < len(A) and A[L].freq < A[i].freq:
      smallest = L
    else:
      smallest = i
    
    if r < len(A) and A[r].freq < A[smallest].freq:
      smallest = r
    
    if(smallest != i):
      temp = A[smallest]
      A[smallest]=A[i]
      A[i]=temp
      minHeapify(A,smallest)

#min-heap minimum
def minHeapMinimum(A):
  if (len(A)>0):
    return A[1]
  else:
    return -1

#min-heap extract min
def minHeapExtractMin(A):
  min = minHeapMinimum(A)
  A[1]=A[-1]
  A.pop()
  minHeapify(A,1)
  return min

#build min-heap
def buildMinHeap(A,n):
  n=int(n/2)
  for i in range(0,n):
    minHeapify(A,n-i)

# Huffman Coding for Image Compression

# Step 2: Build the Huffman Tree
def huffmanTree(min_heap):  
  while(len(min_heap)>2):
      x = minHeapExtractMin(min_heap)
      y = minHeapExtractMin(min_heap)
      z = Node(None,x.freq+y.freq)
      z.left = x
      z.right = y
      insertToMinHeap(min_heap,z)

def assignCodes(node, code, huffmanCodes):
      if node.key is not None:
          # Leaf node
          huffmanCodes[node.key] = code
      else:
          # Internal node
          assignCodes(node.left, code + '0',huffmanCodes)
          assignCodes(node.right, code + '1',huffmanCodes)


# Step 3: Assign binary codes to each leaf node
def huffmanCoding(min_heap): 
  huffmanCodes = dict()
  assignCodes(min_heap[1], '',huffmanCodes)
  return huffmanCodes

# Step 4: Compress the image using the Huffman codes
def image_compression_str(img,huffmanCodes):   
  rows,cols,dims = img.shape
  compressed_image_str = ''
  for i in range(rows):
      for j in range(cols):
          for k in range(dims):
              compressed_image_str += huffmanCodes[img[i,j,k]]
  return compressed_image_str

def save_compressed_file(encoded_image, output_file):    
    extra_padding = 8 - len(encoded_image) % 8
    for i in range(extra_padding):
        encoded_image += "0"

    padded_info = "{0:08b}".format(extra_padding)
    encoded_image = padded_info + encoded_image

    bits = bitarray.bitarray(encoded_image)

    with gzip.open(output_file, 'wb') as f:
        bits.tofile(f)
  
def binToInt(binary):
      sum=0
      i=1
      for j in binary[::-1]:
          val = int(j)*i
          if(val>0):
              sum +=int(j)*i
          i=i*2
      return sum

def compress_img(imagepath,output_file):
    img = cv2.imread(imagepath)
    pixels_dict = pixelDict(img)
    min_heap = make_minHeap(pixels_dict)
    buildMinHeap(min_heap,len(min_heap))
    huffmanTree(min_heap)
    huffmanCodes = huffmanCoding(min_heap)
    compressed_img_str = image_compression_str(img,huffmanCodes)
    save_compressed_file(compressed_img_str,output_file)
    return True


def decompress_img(input_file,decompress_file):
  with gzip.open(input_file, 'rb') as f:
    bits = bitarray.bitarray()
    bits.fromfile(f)
  # convert the bitarray to a string of bits
  bit_string = bits.to01()

  padvalue=binToInt(bit_string[0:8])

  actual_encoded_img = bit_string[8:-1*padvalue]
  print(actual_encoded_img)

while(1):
  print("Enter\n1)To compress an image\n2)To decompress an image\n3)Any other to exit!")
  option=int(input())
  if(option == 1):
    imgname=input("Enter the Image path:")
    filename=input("Enter the location to store the compressed image:")
    compress_img(imgname,filename)
  elif(option == 2):
    imgname=input("Enter the location of compressed image:")
    filename=input("Enter the location to store the decompressed image:")
    decompress_img(imgname,filename)
  else:
    break


Enter
1)To compress an image
2)To decompress an image
3)Any other to exit!


ValueError: invalid literal for int() with base 10: ''

In [1]:

# def pad_encoded_image(encoded_image):
#         extra_padding = 8 - len(encoded_image) % 8
#         for i in range(extra_padding):
#             encoded_image += "0"
#         padded_info = "{0:08b}".format(extra_padding)
#         encoded_image = padded_info + encoded_image
#         return encoded_image
# encoded_image = pad_encoded_image(compressed_image)
# def get_byte_array(padded_encoded_image):
#     if(len(padded_encoded_image) % 8 != 0):
#         print("Encoded image not padded properly")
#         exit(0)
#     b = bytearray()
#     for i in range(0, len(padded_encoded_image), 8):
#         byte = padded_encoded_image[i:i+8]
#         b.append(int(byte, 2))
#     return b

# byte_array = get_byte_array(encoded_image)

# with open("humans1_compressed.bin", 'wb') as output:
#     output.write(byte_array)

# # Step 5: Store the Huffman codes and the compressed image in a file
# with open('huffman_codes.txt', 'w') as file:
#     for key, value in hammingCodes.items():
#         file.write(str(key) + ',' + value + '\n')
# with open('compressed_image.txt', 'w') as file:
#     file.write(compressed_image)

# # Step 6: Read the Huffman codes and the compressed image from the file
# codes = {}
# with open('huffman_codes.txt', 'r') as file:
#     for line in file:
#         key, value = line.strip().split(',')
#         codes[int(key)] = value
# compressed_image = ''
# with open('compressed_image.txt', 'r') as file:
#     compressed_image = file.read()

# # Step 7: Decompress the image using the Huffman codes
# decoded_image = np.zeros((rows,cols,dims), np.uint8)
# code = ''
# k = 0
# for i in range(rows):
#     for j in range(cols):
#         for p in range(dims):
#             while(1):
#               code += compressed_image[k]
#               if code in codes.values():
#                   index = list(codes.values()).index(code)
#                   decoded_image[i,j,p] = list(codes.keys())[index]
#                   code = ''
#                   k +=1
#                   break
#               k += 1
# # Step 8: Display the original and decoded images
# plt.subplot(1, 2, 1)
# plt.title('Original Image')
# plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
# plt.subplot(1, 2, 2)
# plt.title('Decoded Image')
# plt.imshow(cv2.cvtColor(decoded_image, cv2.COLOR_BGR2RGB))
# plt.show()

# # Step 9: Write the decoded image to a file
# cv2.imwrite('decoded_image.jpg', decoded_image)

In [45]:

extra_padding = 8 - len(encoded_image) % 8
for i in range(extra_padding):
    encoded_image += "0"
padded_info = "{0:08b}".format(extra_padding)
encoded_image1 = padded_info + encoded_image
print(padded_info,encoded_image1)

bits = bitarray.bitarray(encoded_image1)

with open('bitImg.bin', 'wb') as f:
    bits.tofile(f)

with open('bitImg.bin', 'rb') as f:
    bits = bitarray.bitarray()
    bits.fromfile(f)
# convert the bitarray to a string of bits
bit_string = bits.to01()

def binToInt(binary):
    sum=0
    i=1
    for j in binary[::-1]:
        val = int(j)*i
        if(val>0):
            sum +=int(j)*i
        i=i*2
    return sum

padvalue=binToInt(bit_string[0:8])

actual_encoded_img = bit_string[8:-1*padvalue]

11100000000111111000000000111111110000000000
00000100 00000100111000000001111110000000001111111100000000000000
bitarray('00000100111000000001111110000000001111111100000000000000')
